In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# =========================
# 1. LIST PAGE URL
# =========================
BASE_LIST_URL = "https://immovlan.be/en/real-estate?transactiontypes=for-sale,in-public-sale&page={}"
# base = "https://immovlan.be/en/real-estate"

# params = {
#     "transactiontypes": "for-sale",
#     "propertytypes": "house,apartment",
#     "propertysubtypes": "residence,villa,mixed-building,master-house,bungalow,cottage,chalet,mansion,apartment,ground-floor,penthouse,duplex,studio,loft,triplex",
#     "islifeannuity": "no",
#     "noindex": "1"
# }

# url = base + "?" + "&".join([f"{k}={v}" for k, v in params.items()])

# print(url)

def get_listing_links(page):
    url = BASE_LIST_URL.format(page)
    r = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(r.text, "lxml")
    links = []

    for a in soup.select("a[href]"):
        href = a.get("href")

        if href and "/en/detail/" in href:
            # full_url = "https://immovlan.be" + href
            links.append(href)

    print("page:", page, "links:", len(links))
    return list(set(links))


# =========================
# 3. COLLECT ALL LISTING LINKS
# =========================
all_links = []

for page in range(1, 3):  # increase range if needed to reach 10k records
    try:
        links = get_listing_links(page)
        all_links.extend(links)
        print(f"Page {page} -> total links collected: {len(all_links)}")
        time.sleep(1)
    except:
        continue

# remove duplicate URLs
print("all_links")
print(all_links)
all_links = list(set(all_links))
print("remove duplicate URLs all_links")
print(all_links)




page: 1 links: 60
Page 1 -> total links collected: 15
page: 2 links: 80
Page 2 -> total links collected: 35
all_links
['https://immovlan.be/en/detail/residence/for-sale/9600/ronse/rbw20337', 'https://immovlan.be/en/detail/residence/for-sale/5640/mettet/vbe35132', 'https://immovlan.be/en/detail/duplex/for-sale/1000/brussels/vbe35095', 'https://immovlan.be/en/detail/commercial-building/for-sale/6000/charleroi/vwd17838', 'https://immovlan.be/en/detail/apartment/for-sale/1210/sint-joost-ten-node/vbe35094', 'https://immovlan.be/en/detail/apartment/for-sale/2390/oostmalle/rbw20430', 'https://immovlan.be/en/detail/residence/for-sale/5300/andenne/vbe35093', 'https://immovlan.be/en/detail/apartment/for-sale/1050/elsene/vbe35040', 'https://immovlan.be/en/detail/duplex/for-sale/7134/epinois/vbe35013', 'https://immovlan.be/en/detail/penthouse/for-sale/2500/lier/rbw20388', 'https://immovlan.be/en/detail/residence/for-sale/6183/trazegnies/vbe35167', 'https://immovlan.be/en/detail/residence/for-sale/

In [18]:

import re

def clean_text(text: str) -> str:
  """Clean the extracted text by removing extra whitespace and special characters.
  Args:        
    text (str): The text to clean.
  Returns:     
    str: The cleaned text.
  """
  # Remove references/citations
  # Examples: [1], [12], [a], [citation needed]
  text = re.sub(r"\[[^\]]*\]", "", text)

  # Remove IPA / phonetic pronunciation blocks
  # Example: (/bəˈrɑːk oʊˈbɑːmə/)
  text = re.sub(r"\([^)]*[/ˈˌ][^)]*\)", "", text)

  # Replace common HTML entities
  text = re.sub(r"&nbsp;", " ", text)
  text = re.sub(r"&amp;", "&", text)

  # Replace multiple spaces, tabs and line breaks with a single space
  text = re.sub(r"\s+", " ", text)

  # Remove spaces before punctuation marks
  # Example: "hello ." -> "hello."
  text = re.sub(r"\s+([.,;:!?])", r"\1", text)

  return text.strip() 

In [36]:

# =========================
# 2. PROPERTY DETAIL SCRAPER
# =========================
def parse_more_info(more_info):
    if more_info is None:
        return {
            field: ""
            for field in FIELD_MAP.keys()
        }
    
    more_info_titles = [h.text.replace("\n", "").strip() for h in more_info.find_all("h4")]
    more_info_contents = [p.text.replace("\n", "").strip() for p in more_info.find_all("p")]
    
    raw_dict = dict(zip(more_info_titles, more_info_contents))

    FIELD_MAP = {
        "property_state": "State of the property",
        "is_rented": "Currently leased",
        "build_year": "Build Year",
        "num_bedrooms": "Number of bedrooms",
        "livable_surface": "Livable surface",
        "furnished": "Furnished",
        "kitchen_equipment": "Kitchen equipment",
        "kitchen_type": "Kitchen type",
        "num_bathrooms": "Number of bathrooms",
        "num_showers": "Number of showers",
        "num_toilets": "Number of toilets",
        "heating_type": "Type of heating",
        "glazing_type": "Type of glazing",
        "elevator": "Elevator",
        "num_facades": "Number of facades",
        "num_floors": "Number of floors",
        "orientation": "Orientation of the front facade",
        "garden": "Garden",
        "terrace": "Terrace",
        "land_surface": "Total land surface",
        "primary_energy_consumption": "Specific primary energy consumption",
        "planning_permission_granted": "Planning permission granted",
        "g_score": "G-score",
        "p_score": "P-score",
        "swimming_pool": "Swimming pool",
        "cellar": "Cellar",
        "veranda": "Veranda",
        "dining_room": "Dining room",
        "attic": "Attic",
        "co2_emission": "CO2 emission",
        "epc_validity_date": "Validity date EPC/PEB",
        "peb_category": "PEB category",
        "latitude": "Latitude",
        "longitude": "Longitude",
        "province": "Province",
    }
        
    cleaned_more_info = {
        field_name: raw_dict.get(page_label, "")
        for field_name, page_label in FIELD_MAP.items()
    }

    return cleaned_more_info

def parse_property(url):
    r = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(r.text, "lxml")
    info = {}

    content = soup.find("div", id="main_content")
    header = content.find("div", class_="detail__header_title")
    info['property_id'] = header.find("span", class_="vlancode").get_text(strip=True)
    info['url'] = url
    info["name"] = header.find("span", class_="detail__header_title_main").get_text(" ", strip=True).rsplit(" ", 1)[0]

    address_info = header.find("div", class_="detail__header_address")
    info["address"] = address_info.find_all("span")[0].get_text(strip=True)
    city = address_info.find("span", class_="city-line").get_text(strip=True)
    info["postcode"], info["city"] = city.split(" ", 1)

    description = content.find("div", class_="dynamic-description").get_text(strip=True)
    info["description"] = clean_text(description)

    financial = content.select_one("div.financial")
    price_li = financial.find("strong", string="Price").parent
    price_text = price_li.get_text(" ", strip=True)
    info["price"] = re.sub(r"[^\d]", "", price_text)

    more_info = content.find("div", class_="general-info-wrapper")
    info.update(parse_more_info(more_info))


    # print(info)
    return info

# =========================
# 4. SCRAPE PROPERTY DETAILS
# =========================
dataset = []

# for url in tqdm(all_links[:10]):  # limit to 10k records
for url in all_links:  # limit to 10k records
    try:
        data = parse_property(url)
        dataset.append(data)
        time.sleep(0.5)  # prevent blocking
    except:
        continue

print("dataset")
print(dataset)


dataset
[{'property_id': 'RBW20337', 'url': 'https://immovlan.be/en/detail/residence/for-sale/9600/ronse/rbw20337', 'name': 'Residence for sale - Ronse', 'address': 'Elzelestraat 43', 'postcode': '9600', 'city': 'Ronse', 'description': 'Ontdek deze instapklare woning met 3 slpk en tuin, gelegen nabij het centrum van Ronse en de Stadstuin.De indeling is als volgt:Gelijkvloers: leefruimte, gang, keuken, wc, bergingVerdieping 1: nachthal met WC en één ruime slaapkamerverdieping 2: 2 slaapkamersTuin: een grote diepe tuinDe troeven van deze woning?- centrale ligging- goedgekeurde elektriciteit- vernieuwde daken- CV op aardgas: ook CV op de zolderkamers!- Overal dubbele beglazingDe woning is momenteel VERHUURD aan 750 € per maand en is dus ook interessant als rendabele investering!Wil je een bezoek brengen aan deze woning? Contacteer me:ContactShow allShow less', 'price': '189000', 'property_state': 'Normal', 'is_rented': '', 'build_year': '', 'num_bedrooms': '3', 'livable_surface': '135 m²'

In [ ]:
import os
import json

def to_json_file(data: dict, filepath: str) -> None:
  """Save the data to a JSON file.
  Args:        
    data (dict): The data to save.
    filename (str): The name of the file to save to.
  Returns:     
    None
  """
  with open(filepath, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)


# base_dir = os.path.dirname(__file__)
base_dir = os.getcwd()
output_filepath = os.path.join(base_dir, "dev/lien_belgium_properties.json")
to_json_file(dataset, output_filepath)

In [ ]:

# =========================
# 5. SAVE DATASET
# =========================
# df = pd.DataFrame(dataset)
# df.to_csv("belgium_properties.csv", index=False)


# print("DONE -> dataset saved with 10k+ properties")